<a href="https://colab.research.google.com/github/bhagath-ac07/AI_learning/blob/main/Transformer_NLP_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Import the necessary Libraries
import re
import pandas as pd
import torch
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, logging
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
logging.set_verbosity_error()

# 1. Load Train and Test data
train_df = pd.read_csv('/content/Train.csv')
test_df = pd.read_csv('/content/Test.csv')

train_df.head(5)
print(f"Total rows in dataset = {len(train_df)} \n")                                        # total rows in dataset
print(f"Total negative and positive in dataset {train_df['label'].value_counts()} \n")
# Check for GPU availability
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")     # total unqiue values of label
# 2. Clean text
def clean_text(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df["cleaned_text"] = train_df["text"].apply(clean_text)
test_df["cleaned_text"] = test_df["text"].apply(clean_text)



# 3. Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class TweetDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer(df["cleaned_text"].tolist(), truncation=True, padding=True)
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(train_df)
test_dataset = TweetDataset(test_df)


# 4. Load model
num_labels = len(set(train_df["label"]) | set(test_df["label"]))
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

# 5. Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,    # Increase batch size if memory allows
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none"
)

# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

# 7. Train
trainer.train()

# 8. Evaluate
predictions = trainer.predict(test_dataset)
preds = torch.argmax(torch.tensor(predictions.predictions), axis=1)

# 9. Report
print("\nClassification Report:\n")
print(classification_report(test_df["label"], preds))


# Convert predictions and actual labels to lists
predicted_labels = preds.tolist()
actual_labels = test_df["label"].tolist()

# Compare in a DataFrame
comparison_df = pd.DataFrame({
    "text": test_df["cleaned_text"].tolist(),
    "actual": actual_labels,
    "predicted": predicted_labels
})

# Print a sample comparison
comparison_df.head(20)


Total rows in dataset = 500 

Total negative and positive in dataset label
0    260
1    240
Name: count, dtype: int64 

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'loss': '0.7101', 'grad_norm': '4.867', 'learning_rate': '4.286e-05', 'epoch': '0.1587'}
{'loss': '0.7115', 'grad_norm': '2.846', 'learning_rate': '3.492e-05', 'epoch': '0.3175'}
{'loss': '0.694', 'grad_norm': '11.11', 'learning_rate': '2.698e-05', 'epoch': '0.4762'}
{'loss': '0.6469', 'grad_norm': '6.968', 'learning_rate': '1.905e-05', 'epoch': '0.6349'}
{'loss': '0.5395', 'grad_norm': '7.26', 'learning_rate': '1.111e-05', 'epoch': '0.7937'}
{'loss': '0.44', 'grad_norm': '7.958', 'learning_rate': '3.175e-06', 'epoch': '0.9524'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '51.61', 'train_samples_per_second': '9.689', 'train_steps_per_second': '1.221', 'train_loss': '0.6157', 'epoch': '1'}

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.93      0.88        85
           1       0.89      0.75      0.82        65

    accuracy                           0.85       150
   macro avg       0.86      0.84      0.85       150
weighted avg       0.86      0.85      0.85       150



,text,actual,predicted
0,I always wrote this series off as being a comp...,0,0
1,1st watched 1272002 3 out of 10DirSteve Purcel...,0,0
2,This movie was so poorly written and directed ...,0,0
3,The most interesting thing about Miryang Secre...,1,1
4,when i first read about berlin am meer i didnt...,0,0
5,I saw this film on September 1st 2005 in India...,1,1
6,I saw a screening of this movie last night I h...,0,1
7,William Hurt may not be an American matinee id...,1,0
8,IT IS A PIECE OF CRAP not funny at all during ...,0,0
9,IM BOUT IT1997br br Developed published by No ...,0,0
